# Dice por diente de los 4 segmentadores sin terceros molares (etapa 1, análisis de sensibilidad)
**Settings:** **CPU**, Internet OFF (no instala nada). Sin re-entrenar ni re-inferir.
**Inputs:**
- salida de `benchmark-segmentacion-cv-nino`: `results/<modelo>/fold<k>_preds.npz` de los 4 modelos (YOLO pintado con conf ≥ 0,25) y `results/metricas_ic95.csv`;
- salida de `datasets-limpios-icdas-segmentacion`: `segmentation_manifest.csv` y `fdi_segmentation_curated.zip`;
- salida de `map-segmentadores`: `map/instancias.pkl.gz` (instancias crudas de YOLO, para YOLO crudo).

Especificación: `claude/encargo_dice_sin_terceros_molares.md`. Celda de datos, decodificador RLE y regla cruda copiados de `dice-yolo-crudo` (`claude/nb_dice_yolo_crudo_celdas.py`).

**Escenarios:** SegFormer-B2, DeepLabV3+, U-Net++, YOLO pintado (benchmark) y YOLO crudo (instancia de mayor confianza con conf ≥ 0,25 y ≥ 50 px, sin pintar).

**Métricas (definiciones del benchmark):** filas foto × clase con la clase en la referencia o predicha con ≥ 50 px; Dice por clase sumando intersección y áreas sobre las fotos.
- **Principal: `dice_28`**, promedio sobre las 28 clases FDI sin terceros molares (sin 18, 28, 38, 48). Los píxeles predichos como tercer molar solo cuentan para su propia clase, que queda fuera del promedio; no se reasignan.
- `dice_32`: promedio sobre las 32 clases (la cifra del benchmark).
- Complementarios: Dice ponderado por diente (media del Dice de cada diente de referencia), con 32 y con 28 clases (`wdice_32`, `wdice_28`).
- Solo terceros molares: Dice de cada una de las 4 clases y su promedio (`dice_3M`).

Bootstrap de niños: 2.000 remuestreos, `default_rng(2026).integers(...)` (los mismos que `map-segmentadores`, `iou-caja-segmentadores` y `dice-yolo-crudo`); media ± DE por fold.

In [ ]:
import os, glob, gzip, pickle, time, zipfile, re
import numpy as np, pandas as pd, cv2
from concurrent.futures import ThreadPoolExecutor
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
MODELS = ['segformer_b2', 'deeplabv3p_r50', 'unetpp_r50', 'yolo11m_seg']
LABEL = {'segformer_b2': 'SegFormer-B2', 'deeplabv3p_r50': 'DeepLabV3+', 'unetpp_r50': 'U-Net++', 'yolo11m_seg': 'YOLO pintado'}
ESC = ['SegFormer-B2', 'DeepLabV3+', 'U-Net++', 'YOLO pintado', 'YOLO crudo']
M3 = {18, 28, 38, 48}
MIN_PX, CONF = 50, 0.25
B, SEED = 2000, 2026
TOL = 0.001
N_REF, N_3M, N_SIN3M, N_INST, N_INST025 = 9710, 102, 9608, 14382, 10130
CRUDO_32 = 0.880                     # resultados_dice_yolo_crudo.md (3 decimales)
IN, W, TMP = '/kaggle/input', '/kaggle/working', '/kaggle/tmp'
OUT = f'{W}/dice_sin_3M'; os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
T0 = time.time()

## Datos, labelmap y FDI (igual que en el benchmark)
Solo se leen las máscaras de referencia (no las fotos).

In [ ]:
def find(name):
    h = sorted(glob.glob(f'{IN}/**/{name}', recursive=True)); return h[0] if h else None
man = pd.read_csv(find('segmentation_manifest.csv'))
DATA = f'{TMP}/seg'
if not glob.glob(f'{DATA}/**/{man.mask_file[0]}', recursive=True):
    zipfile.ZipFile(find('fdi_segmentation_curated.zip')).extractall(DATA)
ROOT = os.path.dirname(os.path.dirname(glob.glob(f'{DATA}/**/{man.mask_file[0]}', recursive=True)[0]))
LM = (glob.glob(f'{DATA}/**/labelmap.txt', recursive=True) or [find('labelmap.txt')])[0]
lines = [l.strip() for l in open(LM, encoding='utf-8') if l.strip() and not l.startswith('#')]
NAMES, COLORS = [], []
for l in lines:
    p = l.split(':'); NAMES.append(p[0].strip()); COLORS.append(tuple(int(v) for v in p[1].split(',')) if len(p) > 1 and p[1] else None)
bg = [i for i, n in enumerate(NAMES) if n.lower() in ('background', 'fondo', 'bg')]
if bg and bg[0] != 0:
    i = bg[0]; NAMES.insert(0, NAMES.pop(i)); COLORS.insert(0, COLORS.pop(i))
NC = len(NAMES)
m0 = cv2.imread(os.path.join(ROOT, man.mask_file[0]), cv2.IMREAD_UNCHANGED); RGB_MASK = m0.ndim == 3
LUT = {c: i for i, c in enumerate(COLORS) if c is not None}
def read_mask(rel):
    m = cv2.imread(os.path.join(ROOT, rel), cv2.IMREAD_UNCHANGED)
    if not RGB_MASK: return m.astype(np.uint8)
    m = cv2.cvtColor(m[..., :3], cv2.COLOR_BGR2RGB); key = (m[..., 0].astype(np.int32) << 16) | (m[..., 1].astype(np.int32) << 8) | m[..., 2]
    out = np.zeros(m.shape[:2], np.uint8); seen = set(np.unique(key))
    for c, i in LUT.items():
        k = (c[0] << 16) | (c[1] << 8) | c[2]
        if k in seen: out[key == k] = i
    return out
with ThreadPoolExecutor(8) as ex:
    MASKS = dict(zip(man.image_id, ex.map(read_mask, man.mask_file)))
def univ(n):
    m = re.search(r'\d+', n); return int(m.group()) if m else None
def to_fdi(u):
    if u is None or not 1 <= u <= 32: return None
    if u <= 8: return 10 + (9 - u)
    if u <= 16: return 20 + (u - 8)
    if u <= 24: return 30 + (25 - u)
    return 40 + (u - 24)
U = [univ(n) for n in NAMES]; FDI = [to_fdi(u) if i else None for i, u in enumerate(U)]
assert NC == 33 and all(f is not None for f in FDI[1:])
TEETH = list(range(1, NC))
assert man.groupby('child_id').fold.nunique().max() == 1
def grupo(f):
    u = f % 10
    return {1: 'incisivos', 2: 'incisivos', 3: 'caninos', 4: 'premolares', 5: 'premolares', 6: 'molares 1.º–2.º', 7: 'molares 1.º–2.º', 8: 'terceros molares'}[u]
GRUPOS = ['incisivos', 'caninos', 'premolares', 'molares 1.º–2.º', 'terceros molares']
RES = os.path.dirname(find('metricas_ic95.csv')); REF = pd.read_csv(f'{RES}/metricas_ic95.csv').set_index('model')
for m in MODELS:
    for k in range(5): assert os.path.exists(f'{RES}/{m}/fold{k}_preds.npz'), f'falta {RES}/{m}/fold{k}_preds.npz'
print(len(MASKS), 'fotos |', man.child_id.nunique(), 'niños | clases', NC, '| raíz', ROOT, '| predicciones:', RES)
print(man.groupby('fold').agg(fotos=('image_id', 'size'), ninos=('child_id', 'nunique')).T.to_string())
print(REF[['dice_tooth', 'id_f1']].round(3).to_string())

## Caché de instancias crudas de YOLO y decodificador RLE (copiados de `dice-yolo-crudo`)

In [ ]:
def rle_counts(s):
    """Decodifica el string comprimido de COCO (igual que rleFrString de pycocotools)."""
    if isinstance(s, list): return s
    if isinstance(s, str): s = s.encode()
    cnts, p = [], 0
    while p < len(s):
        x, k, more = 0, 0, 1
        while more:
            c = s[p] - 48; x |= (c & 0x1f) << 5 * k; more = c & 0x20; p += 1; k += 1
            if not more and (c & 0x10): x |= -1 << 5 * k
        if len(cnts) > 2: x += cnts[-2]
        cnts.append(x)
    return cnts
def rle_decode(r):
    h, w = r['size']; cnts = np.asarray(rle_counts(r['counts']), np.int64)
    v = np.repeat(np.arange(len(cnts)) % 2 == 1, cnts)
    assert v.size == h * w, 'RLE con largo distinto de h*w'
    return v.reshape(w, h).T

INST = pickle.load(gzip.open(find('instancias.pkl.gz'))); Y = INST['yolo11m_seg']; del INST
assert len(Y) == len(man) and all(j in Y for j in man.image_id), 'el caché no tiene todas las fotos'
n_inst = sum(len(Y[j]['main']) for j in Y); n_inst025 = sum(s >= CONF for j in Y for _, s, _ in Y[j]['main'])
dec_msg = 'pycocotools no instalado: sin control cruzado'
try:
    from pycocotools import mask as MU
    smp = [q for j in list(Y)[:10] for _, _, q in Y[j]['main'][:4]]
    ok = all(np.array_equal(rle_decode(q), MU.decode(q).astype(bool)) for q in smp)
    dec_msg = f'decodificador vs pycocotools.decode en {len(smp)} máscaras: {"coincide" if ok else "NO COINCIDE"}'; assert ok
except ImportError: pass
print(f'instancias crudas de YOLO: {n_inst} (esperado {N_INST}); con conf ≥ {CONF}: {n_inst025} (esperado {N_INST025}) | {dec_msg}')

## Tabla por foto × clase para los 5 escenarios
Para YOLO crudo, por foto y clase: la instancia de mayor confianza con conf ≥ 0,25 y ≥ 50 px, entera (misma regla que `dice-yolo-crudo` y que S2 del IoU de caja).

In [ ]:
IDX = {f: c for c, f in enumerate(FDI) if f is not None}
def raw_best(j):
    by = {}
    for f, s, q in Y[j]['main']:
        if s >= CONF: by.setdefault(IDX[int(f)], []).append((s, q))
    out = {}
    for c, L in by.items():
        for _, q in sorted(L, key=lambda x: -x[0]):
            m = rle_decode(q)
            if m.sum() >= MIN_PX: out[c] = m; break
    return out

rows = []
t = time.time()
for k in range(5):
    P = {m: np.load(f'{RES}/{m}/fold{k}_preds.npz') for m in MODELS}
    for j in man[man.fold == k].image_id:
        g = MASKS[j]; cg = np.bincount(g.ravel(), minlength=NC)[:NC]
        SC = {}
        for m in MODELS:
            p = P[m][str(j)]; assert p.shape == g.shape, f'{m} {j}: forma distinta'
            cp = np.bincount(p.ravel(), minlength=NC)[:NC]
            SC[LABEL[m]] = {c: p == c for c in TEETH if cp[c] > 0}
        SC['YOLO crudo'] = raw_best(j)
        for esc, Mk in SC.items():
            for c in TEETH:
                Pm = Mk.get(c); npx = int(Pm.sum()) if Pm is not None else 0
                if cg[c] == 0 and npx < MIN_PX: continue
                inter = int(((g == c) & Pm).sum()) if Pm is not None else 0
                rows.append((esc, j, FDI[c], int(cg[c]), npx, inter))
    del P
print(f'tabla: {time.time() - t:.0f} s')
T = pd.DataFrame(rows, columns=['escenario', 'image_id', 'fdi', 'gt_px', 'pred_px', 'inter'])
T = T.merge(man[['image_id', 'child_id', 'fold']], on='image_id', how='left')
T['m3'] = T.fdi.isin(M3); T['in_gt'] = T.gt_px > 0
T['dice'] = 2 * T.inter / (T.gt_px + T.pred_px).clip(lower=1)
print(T.groupby('escenario').agg(filas=('fdi', 'size'), dientes_ref=('in_gt', 'sum')).to_string())

## Métricas y chequeos (si fallan, el notebook se detiene)
- Con 32 clases se reproduce el Dice por diente de `metricas_ic95.csv` para los 4 modelos (±0,001).
- YOLO crudo con 32 clases: 0,880.
- Dientes de referencia: 9.710; terceros molares: 102; sin ellos: 9.608. Instancias del caché: 14.382 (10.130 con conf ≥ 0,25).

In [ ]:
def dice_cls(d):
    """Dice por clase (FDI) con la definición del benchmark."""
    pc = d.groupby('fdi').agg(I=('inter', 'sum'), G=('gt_px', 'sum'), Pp=('pred_px', 'sum'))
    S = pc.G + pc.Pp; return (2 * pc.I / S)[S > 0]
def mets(d):
    dc = dice_cls(d); g = d[d.in_gt]
    return dict(dice_32=dc.mean(), dice_28=dc[~dc.index.isin(M3)].mean(), dice_3M=dc[dc.index.isin(M3)].mean(),
                wdice_32=g.dice.mean(), wdice_28=g[~g.m3].dice.mean(),
                n_ref=int(len(g)), n_ref_28=int((~g.m3).sum()), n_ref_3M=int(g.m3.sum()))
PT = {e: mets(T[T.escenario == e]) for e in ESC}
chk = [dict(chequeo=f'Dice por diente (32 clases), {LABEL[m]}', valor=PT[LABEL[m]]['dice_32'], esperado=REF.loc[m, 'dice_tooth'], tol=TOL) for m in MODELS]
chk += [dict(chequeo='Dice por diente (32 clases), YOLO crudo', valor=PT['YOLO crudo']['dice_32'], esperado=CRUDO_32, tol=0.0006),
        dict(chequeo='dientes de referencia', valor=PT['SegFormer-B2']['n_ref'], esperado=N_REF, tol=0),
        dict(chequeo='terceros molares de referencia', valor=PT['SegFormer-B2']['n_ref_3M'], esperado=N_3M, tol=0),
        dict(chequeo='dientes de referencia sin terceros molares', valor=PT['SegFormer-B2']['n_ref_28'], esperado=N_SIN3M, tol=0),
        dict(chequeo='instancias crudas de YOLO en el caché', valor=n_inst, esperado=N_INST, tol=0),
        dict(chequeo='instancias con conf >= 0,25', valor=n_inst025, esperado=N_INST025, tol=0)]
chk = pd.DataFrame(chk); chk['ok'] = (chk.valor - chk.esperado).abs() <= chk.tol
chk.to_csv(f'{OUT}/dice_sin_3M_chequeos.csv', index=False); print(chk.to_string(index=False))
assert chk.ok.all(), 'ALGÚN CHEQUEO FALLA: se detiene (ver dice_sin_3M_chequeos.csv)'
print('chequeos OK')

## Resumen: 32 clases, 28 clases y ponderado por diente; IC 95 % por bootstrap de niños y media ± DE por fold
Bootstrap con sumas por niño y clase (equivale a concatenar los niños remuestreados).

In [ ]:
KIDS = np.array(sorted(man.child_id.unique())); KI = {c: i for i, c in enumerate(KIDS)}
rng = np.random.default_rng(SEED)
DRAWS = rng.integers(0, len(KIDS), size=(B, len(KIDS)))      # idéntico a los notebooks anteriores
WK = np.stack([np.bincount(r, minlength=len(KIDS)) for r in DRAWS]).astype(float)
FD = sorted(f for f in FDI if f is not None); FI = {f: i for i, f in enumerate(FD)}; IS3 = np.array([f in M3 for f in FD])
def boot(d):
    kid = d.child_id.map(KI).to_numpy(); cl = d.fdi.map(FI).to_numpy()
    def m(v):
        a = np.zeros((len(KIDS), len(FD))); np.add.at(a, (kid, cl), v); return WK @ a
    I = m(d.inter.to_numpy(float)); S = m((d.gt_px + d.pred_px).to_numpy(float))
    with np.errstate(invalid='ignore', divide='ignore'):
        D = np.where(S > 0, 2 * I / S, np.nan)
    def c(v, sel):
        return WK @ np.bincount(kid[sel], weights=v[sel], minlength=len(KIDS))
    g = d.in_gt.to_numpy(); m3 = d.m3.to_numpy(); dv = d.dice.to_numpy(float); one = np.ones(len(d))
    with np.errstate(invalid='ignore', divide='ignore'):
        r = dict(dice_32=np.nanmean(D, 1), dice_28=np.nanmean(D[:, ~IS3], 1), dice_3M=np.nanmean(D[:, IS3], 1),
                 wdice_32=c(dv, g) / c(one, g), wdice_28=c(dv, g & ~m3) / c(one, g & ~m3))
    return pd.DataFrame(r), pd.DataFrame(D[:, IS3], columns=[f for f in FD if f in M3])
MET = ['dice_28', 'dice_32', 'wdice_28', 'wdice_32', 'dice_3M']
BOOT, B3M, res, pf_rows = {}, {}, [], []
for e in ESC:
    d = T[T.escenario == e]; bs, b3 = boot(d); BOOT[e] = bs; B3M[e] = b3
    pf = pd.DataFrame([dict(fold=k, **mets(x)) for k, x in d.groupby('fold')]); pf_rows += [dict(escenario=e, **r) for r in pf.to_dict('records')]
    row = dict(escenario=e, **PT[e])
    for m_ in MET:
        row[f'{m_}_lo'] = bs[m_].quantile(.025); row[f'{m_}_hi'] = bs[m_].quantile(.975)
        row[f'{m_}_fold_media'] = pf[m_].mean(); row[f'{m_}_fold_de'] = pf[m_].std()
    res.append(row)
res = pd.DataFrame(res); res.to_csv(f'{OUT}/dice_sin_3M_resumen.csv', index=False)
pd.DataFrame(pf_rows).to_csv(f'{OUT}/dice_sin_3M_por_fold.csv', index=False)
print('Dice por diente — combinado fuera de fold [IC 95 %] | por fold media ± DE')
for _, r in res.iterrows():
    print(f'\n{r.escenario}  (dientes ref {r.n_ref}; sin 3M {r.n_ref_28}; 3M {r.n_ref_3M})')
    for m_ in MET: print(f'   {m_:9s} {r[m_]:.4f} [{r[m_ + "_lo"]:.4f}–{r[m_ + "_hi"]:.4f}] | {r[m_ + "_fold_media"]:.4f} ± {r[m_ + "_fold_de"]:.4f}')

## Comparaciones pareadas (bootstrap pareado de niños, mismos remuestreos; Holm dentro de cada métrica)
Los 6 pares de los 4 modelos del benchmark (Table 2) más SegFormer-B2 − YOLO crudo. Métrica principal `dice_28`; también `dice_32` y `wdice_28`.

In [ ]:
PAIRS = [('SegFormer-B2', 'YOLO pintado'), ('SegFormer-B2', 'YOLO crudo'), ('SegFormer-B2', 'DeepLabV3+'), ('SegFormer-B2', 'U-Net++'),
         ('DeepLabV3+', 'U-Net++'), ('DeepLabV3+', 'YOLO pintado'), ('U-Net++', 'YOLO pintado')]
cmp = []
for m_ in ('dice_28', 'dice_32', 'wdice_28'):
    part = []
    for a, b in PAIRS:
        dl = BOOT[a][m_].to_numpy() - BOOT[b][m_].to_numpy()
        part.append(dict(metrica=m_, a=a, b=b, dif=PT[a][m_] - PT[b][m_], lo=np.quantile(dl, .025), hi=np.quantile(dl, .975),
                         p=min(1, 2 * min((dl <= 0).mean(), (dl >= 0).mean()))))
    part = pd.DataFrame(part).sort_values('p').reset_index(drop=True); n = len(part)
    part['p_holm'] = np.minimum(1, np.maximum.accumulate([(n - i) * p for i, p in enumerate(part.p)])); cmp.append(part)
cmp = pd.concat(cmp, ignore_index=True); cmp.to_csv(f'{OUT}/dice_sin_3M_pareada.csv', index=False)
for m_ in ('dice_28', 'dice_32', 'wdice_28'):
    print(f'\n{m_}:'); print(cmp[cmp.metrica == m_].drop(columns='metrica').round(4).to_string(index=False))

## Solo terceros molares: Dice por clase (18, 28, 38, 48) y promedio, con IC y n

In [ ]:
g0 = T[(T.escenario == 'SegFormer-B2') & T.in_gt]
n_cls = g0[g0.m3].groupby('fdi').size(); n_kids_3m = g0[g0.m3].child_id.nunique(); n_img_3m = g0[g0.m3].image_id.nunique()
solo = []
for e in ESC:
    dc = dice_cls(T[T.escenario == e])
    for f in sorted(M3):
        b = B3M[e][f]
        solo.append(dict(escenario=e, fdi=f, n_ref=int(n_cls.get(f, 0)), dice=dc.get(f, np.nan), lo=b.quantile(.025), hi=b.quantile(.975)))
    b = BOOT[e].dice_3M
    solo.append(dict(escenario=e, fdi='promedio 4 clases', n_ref=int(n_cls.sum()), dice=PT[e]['dice_3M'], lo=b.quantile(.025), hi=b.quantile(.975)))
solo = pd.DataFrame(solo); solo['n_ninos_con_3M'] = n_kids_3m; solo['n_fotos_con_3M'] = n_img_3m
solo.to_csv(f'{OUT}/dice_3M_solo.csv', index=False)
print(f'Terceros molares de referencia: {int(n_cls.sum())} ({n_cls.to_dict()}) en {n_img_3m} fotos de {n_kids_3m} niños')
print(solo.pivot_table(index='fdi', columns='escenario', values='dice', sort=False)[ESC].round(3).to_string())

## Lectura automática (para el informe)
Regla del encargo: si el IC de SegFormer-B2 − YOLO sin terceros molares incluye 0 y su semiancho es ≤ 0,01 → "equivalent/comparable"; si incluye 0 y es más ancho → "similar"; si excluye 0, la brecha no viene solo de los terceros molares.

In [ ]:
def lee(a, b):
    r = cmp[(cmp.metrica == 'dice_28') & (cmp.a == a) & (cmp.b == b)].iloc[0]; half = (r.hi - r.lo) / 2
    v = 'equivalente/comparable' if r.lo <= 0 <= r.hi and half <= 0.01 else 'similar (IC incluye 0, ancho)' if r.lo <= 0 <= r.hi else 'la brecha persiste (IC excluye 0)'
    return f'{a} − {b}: Δ dice_28 = {r.dif:+.4f} [{r.lo:+.4f}, {r.hi:+.4f}] (semiancho {half:.4f}; p_holm {r.p_holm:.3f}) → {v}'
for a, b in PAIRS: print(lee(a, b))
print('\ntiempo total', round((time.time() - T0) / 60, 1), 'min'); print(sorted(os.listdir(OUT)))